In [1]:
"""
OSI SAF Sea Ice Index v3.0 (OSI-420-a): Download All Arctic Regions
========================================================================

Downloads Northern Hemisphere sea-ice extent (and optionally area) index
files for ALL available Arctic regions from the newly-operational OSI SAF
Sea Ice Index v3.0 (OSI-420-a, released 4 May 2026), replacing the
previous v2.3 product used elsewhere in this repo (OSISAF_PATH =
'../../osi-sea_ice_index/' in figure_01).

IMPORTANT -- migrating from v2.3 to v3.0
------------------------------------------
The region naming has changed. figure_01_SIE_timeseries_and_SIC_spatial_
trends.ipynb currently calls fetch_sie('fram', ...) for the Greenland Sea
region. In v3.0 the equivalent region is named 'greenland' ("East
Greenland Sea"), NOT 'fram'. If you point figure_01 at this new data,
that region key needs updating, or fetch_sie() calls will silently return
nothing / raise a KeyError depending on how it's called downstream.

URL verification status
------------------------
CONFIRMED DIRECT: all Northern Hemisphere region URLs and the file-naming
pattern below were read directly off the official product page
(https://osisaf-hl.met.no/v3p0-sea-ice-index), not guessed. Both 'nc' and
'txt' formats are listed there for daily and monthly ice_extent per region.

UNVERIFIED BY INFERENCE: the 'ice_area' (SIA) file paths are NOT shown
explicitly in that page's data table (only 'ice_extent' is), but the same
page's image links (e.g. ice_area_nh_sii-v3p0_monthly-prev.png) follow an
identical naming convention to the ice_extent files, strongly suggesting
ice_area_{region}_sii-v3p0_{freq}.nc exists at the parallel path. This is
inferred by pattern consistency, not confirmed by an explicit data-table
link -- if DOWNLOAD_AREA below produces 404s, that inference was wrong;
disable it and fetch SIA a different way.

UNVERIFIED -- internal file structure: thredds.met.no is not reachable
from the sandbox this was drafted in, so the actual NetCDF internal
variable name(s) inside these files could not be inspected. The loader
below is written defensively -- it opens each file, logs whatever data
variable(s) it actually finds, and uses them generically rather than
assuming a specific hardcoded variable name. Check the log on first run.

Regions (Northern Hemisphere / Arctic)
----------------------------------------
Global, Northern Hemisphere, Baffin Bay and Labrador Sea, Baltic Sea,
Barents Sea, Beaufort Sea, Bering Sea, Bohai and Yellow Seas, Canadian
Archipelago, Central Arctic, Chukchi Sea, East Greenland Sea, East
Siberian Sea, Gulf of St. Lawrence, Hudson Bay, Kara Sea, Laptev Sea,
Sea of Japan, Sea of Okhotsk, Svalbard.
(Southern Hemisphere regions exist in the same product but are excluded
by default here -- see INCLUDE_SH below to add them.)

Outputs
-------
  Raw downloads:      ../../osisaf_sii_v3/raw/{var}_{region}_{freq}.nc
  Per-region CSVs:    ./outputs/processed_data/osisaf_sii_v3/{var}_{region}_{freq}.csv
  Combined NetCDF:    ./outputs/processed_data/osisaf_sii_v3/osisaf_sii_v3_all_regions.nc
                       (one variable per region/frequency/quantity, for
                       cross-region comparison)
  Methodology:         ./outputs/methods/osisaf_sii_v3_methodology.md
  Log:                 ./outputs/logs/osisaf_sii_v3.log

Version:       0.1.0 (draft -- ice_area paths and internal variable names unverified)
Last Modified: 2026-08-20
Author:        Chris Barrell
"""

import os
import sys
import logging
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import requests

warnings.filterwarnings("ignore")

sys.path.append('..')
try:
    from utils.logger import get_logger
    logger = get_logger(__name__, config_path='../config.yaml')
except Exception:
    logger = logging.getLogger(__name__)
    logging.basicConfig(level=logging.INFO)

# ============================================================================
# CONFIGURATION
# ============================================================================

RAW_DIR       = Path('../../osisaf_sii_v3/raw/')
PROCESSED_DIR = Path('./outputs/processed_data/osisaf_sii_v3/')
METHODS_DIR   = Path('./outputs/methods/')
LOG_DIR       = Path('./outputs/logs/')
for d in [RAW_DIR, PROCESSED_DIR, METHODS_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

log_path = LOG_DIR / 'osisaf_sii_v3.log'
fh = logging.FileHandler(log_path, mode='w')
fh.setLevel(logging.INFO)
fh.setFormatter(logging.Formatter('%(asctime)s %(levelname)s %(message)s'))
logger.addHandler(fh)

REPROCESS = True  # Set False after first successful run to reuse cached downloads

BASE_URL = 'https://thredds.met.no/thredds/fileServer/osisaf/met.no/ice/index/sii_v3p0'

# --- Region registry: key -> display name ---
NH_REGIONS = {
    'glb':        'Global',
    'nh':         'Northern Hemisphere',
    'baffin':     'Baffin Bay and Labrador Sea',
    'baltic':     'Baltic Sea',
    'barents':    'Barents Sea',
    'beaufort':   'Beaufort Sea',
    'bering':     'Bering Sea',
    'bohai':      'Bohai and Yellow Seas',
    'canarch':    'Canadian Archipelago',
    'centralarc': 'Central Arctic',
    'chukchi':    'Chukchi Sea',
    'greenland':  'East Greenland Sea',   # NOTE: was 'fram' under v2.3 -- see docstring
    'ess':        'East Siberian Sea',
    'lawrence':   'Gulf of St. Lawrence',
    'hudson':     'Hudson Bay',
    'kara':       'Kara Sea',
    'laptev':     'Laptev Sea',
    'japan':      'Sea of Japan',
    'okhotsk':    'Sea of Okhotsk',
    'sval':       'Svalbard',
}

SH_REGIONS = {
    'sh':    'Southern Hemisphere',
    'bell':  'Amundsen-Bellingshausen Seas',
    'drml':  'Dronning Maud Land',
    'indi':  'Indian Ocean',
    'ross':  'Ross Sea',
    'wedd':  'Weddell Sea',
    'wpac':  'Western Pacific Ocean',
}

INCLUDE_SH = False  # set True to also fetch Southern Hemisphere regions

REGIONS = dict(NH_REGIONS)
if INCLUDE_SH:
    REGIONS.update(SH_REGIONS)

# --- What to download ---
VARIABLES  = ['ice_extent']   # add 'ice_area' once confirmed (see docstring)
DOWNLOAD_AREA = False          # set True to also attempt ice_area (unverified paths)
if DOWNLOAD_AREA and 'ice_area' not in VARIABLES:
    VARIABLES.append('ice_area')

FREQUENCIES = ['daily', 'monthly']   # figure_01 uses daily (annual-max extraction);
                                       # monthly is lighter for quick cross-region checks


## Section 1 - Download function

In [2]:
def build_url(var, region, freq):
    return f'{BASE_URL}/{region}/{var}_{region}_sii-v3p0_{freq}.nc'


def download_raw(var, region, freq, raw_dir=RAW_DIR, reprocess=REPROCESS, timeout=60):
    """Download one region/variable/frequency NetCDF file, or reuse cache."""
    url = build_url(var, region, freq)
    raw_path = raw_dir / f'{var}_{region}_{freq}.nc'

    if not reprocess and raw_path.exists():
        logger.info(f"[{region}/{var}/{freq}] Using cached file: {raw_path}")
        return raw_path

    try:
        resp = requests.get(url, timeout=timeout)
        resp.raise_for_status()
    except requests.exceptions.RequestException as e:
        logger.error(f"[{region}/{var}/{freq}] Download FAILED from {url}: {e}")
        return None

    raw_path.write_bytes(resp.content)
    logger.info(f"[{region}/{var}/{freq}] Downloaded {len(resp.content)} bytes -> {raw_path}")
    return raw_path


## Section 2 - Download all regions/variables/frequencies

In [3]:
downloaded = {}
key_meta = {}  # key -> (var, region, freq) -- avoids fragile re-parsing of the
                # concatenated key string later (e.g. 'ice_extent' itself
                # contains an underscore, so splitting the key back apart
                # is ambiguous; keep the structured tuple instead)

for region in REGIONS:
    for var in VARIABLES:
        for freq in FREQUENCIES:
            key = f'{var}_{region}_{freq}'
            raw_path = download_raw(var, region, freq)
            downloaded[key] = raw_path
            key_meta[key] = (var, region, freq)

n_ok = sum(1 for v in downloaded.values() if v is not None)
logger.info(f"Downloaded {n_ok} / {len(downloaded)} files "
            f"({len(REGIONS)} regions x {len(VARIABLES)} variable(s) x {len(FREQUENCIES)} frequencies)")


INFO:__main__:[glb/ice_extent/daily] Downloaded 1485531 bytes -> ../../osisaf_sii_v3/raw/ice_extent_glb_daily.nc
INFO:__main__:[glb/ice_extent/monthly] Downloaded 84969 bytes -> ../../osisaf_sii_v3/raw/ice_extent_glb_monthly.nc
INFO:__main__:[nh/ice_extent/daily] Downloaded 1485531 bytes -> ../../osisaf_sii_v3/raw/ice_extent_nh_daily.nc
INFO:__main__:[nh/ice_extent/monthly] Downloaded 84995 bytes -> ../../osisaf_sii_v3/raw/ice_extent_nh_monthly.nc
INFO:__main__:[baffin/ice_extent/daily] Downloaded 1485531 bytes -> ../../osisaf_sii_v3/raw/ice_extent_baffin_daily.nc
INFO:__main__:[baffin/ice_extent/monthly] Downloaded 85013 bytes -> ../../osisaf_sii_v3/raw/ice_extent_baffin_monthly.nc
INFO:__main__:[baltic/ice_extent/daily] Downloaded 1485531 bytes -> ../../osisaf_sii_v3/raw/ice_extent_baltic_daily.nc
INFO:__main__:[baltic/ice_extent/monthly] Downloaded 84977 bytes -> ../../osisaf_sii_v3/raw/ice_extent_baltic_monthly.nc
INFO:__main__:[barents/ice_extent/daily] Downloaded 1485531 bytes ->

## Section 3 - Load each file generically, log actual variable names found

In [4]:
def load_index_nc(raw_path, key):
    """Open one OSI SAF SII v3 index NetCDF generically -- doesn't assume a
    specific internal variable name (unverified, see docstring). Logs
    whatever data variable(s) it finds so the assumption can be checked."""
    if raw_path is None:
        return None
    try:
        ds = xr.open_dataset(raw_path)
    except Exception as e:
        logger.error(f"[{key}] Failed to open {raw_path}: {e}")
        return None

    data_vars = list(ds.data_vars)
    logger.info(f"[{key}] Opened OK. Data variables found: {data_vars}, "
                f"dims: {dict(ds.sizes)}")

    if not data_vars:
        logger.warning(f"[{key}] No data variables found -- check file contents manually.")
        return None

    return ds


loaded = {}
for key, raw_path in downloaded.items():
    ds = load_index_nc(raw_path, key)
    if ds is not None:
        loaded[key] = ds

logger.info(f"Successfully opened {len(loaded)} / {len(downloaded)} files")


INFO:argopy.stores.ArgoFloat:Using ONLINE Argo Float implementation
INFO:httpx:HTTP Request: GET https://raw.githubusercontent.com/IrishMarineInstitute/awesome-erddap/master/erddaps.json "HTTP/1.1 200 OK"
INFO:__main__:[ice_extent_glb_daily] Opened OK. Data variables found: ['time_bounds', 'lat_bounds', 'lon_bounds', 'area', 'sie', 'source', 'rank_per_doy', 'min_per_doy', 'max_per_doy', 'yearly_min_rank', 'yearly_max_rank', 'yearly_min_date', 'yearly_max_date', 'yearly_min_value', 'yearly_max_value'], dims: {'time': 17466, 'nv': 2, 'dayofyear': 366, 'year': 47}
INFO:__main__:[ice_extent_glb_monthly] Opened OK. Data variables found: ['time_bounds', 'lat_bounds', 'lon_bounds', 'area', 'sie', 'rank_per_month', 'absolute_trend', 'relative_trend_1981_2010', 'relative_trend_1991_2020'], dims: {'time': 583, 'nv': 2, 'month': 12}
INFO:__main__:[ice_extent_nh_daily] Opened OK. Data variables found: ['time_bounds', 'lat_bounds', 'lon_bounds', 'area', 'sie', 'source', 'rank_per_doy', 'min_per_doy

## Section 4 - Per-region CSVs + combined multi-region NetCDF (for cross comparisons)

In [5]:
# Confirmed from real OSI SAF SII v3 files (2026-08-20): 'ice_extent' files
# contain a full CF-style bundle -- time_bounds, lat_bounds, lon_bounds, area,
# sie, source, rank_per_doy/rank_per_month, min/max_per_doy, yearly_min/max_*,
# absolute_trend, relative_trend_1981_2010, relative_trend_1991_2020 -- with
# 'sie' as the actual primary quantity. 'ice_area' files presumably use 'sia'
# by analogy (not yet confirmed against a real file).
PRIMARY_VAR_CANDIDATES = ['sie', 'sia', 'ice_extent', 'ice_area']


def extract_primary_series(ds, key, time_dim='time'):
    """Select the actual primary quantity (e.g. 'sie'), not just the first
    data variable -- real files bundle many auxiliary variables (bounds,
    ranks, trends, source strings) alongside it, and blindly taking
    data_vars[0] can grab something like 'time_bounds' instead, which is
    2-D (time, nbounds) and produces duplicate rows per timestep when
    flattened to a dataframe -- exactly the bug this replaces.

    Restricts candidates to variables whose ONLY dimension is time_dim
    (rules out bounds/aux variables with extra dims), then prefers a
    known primary name from PRIMARY_VAR_CANDIDATES if present.
    """
    data_vars = list(ds.data_vars)
    candidates_1d = [v for v in data_vars if ds[v].dims == (time_dim,)]

    var_name = next((c for c in PRIMARY_VAR_CANDIDATES if c in candidates_1d), None)
    if var_name is None and candidates_1d:
        var_name = candidates_1d[0]
        logger.warning(f"[{key}] No known primary variable name found among "
                        f"1-D {time_dim}-only variables {candidates_1d} -- "
                        f"using '{var_name}'. Verify this is the intended quantity.")
    if var_name is None:
        logger.error(f"[{key}] No 1-D {time_dim}-only variable found among "
                     f"{data_vars} -- cannot extract a primary series (all "
                     f"variables have extra dimensions, e.g. bounds-type). "
                     f"Inspect manually.")
        return None

    if len(data_vars) > 1:
        logger.info(f"[{key}] {len(data_vars)} data variables present "
                    f"({data_vars}); selected '{var_name}' as the primary quantity.")

    da = ds[var_name]
    df = da.to_dataframe().reset_index()
    df = df.rename(columns={time_dim: 'time', var_name: key})

    if df['time'].duplicated().any():
        n_dup = int(df['time'].duplicated().sum())
        logger.warning(f"[{key}] {n_dup} duplicate time values found even after "
                       f"selecting '{var_name}' -- dropping duplicates, keeping first.")
        df = df.drop_duplicates(subset='time', keep='first')

    return df[['time', key]]


combined = None
for key, ds in loaded.items():
    var, region, freq = key_meta[key]
    df = extract_primary_series(ds, key)
    if df is None:
        logger.warning(f"[{key}] Skipped -- no usable primary series extracted.")
        continue
    out_csv = PROCESSED_DIR / f'{key}.csv'
    df.to_csv(out_csv, index=False)

    da_long = df.set_index('time')[key].to_xarray()
    da_ds = da_long.to_dataset(name=key)
    da_ds[key].attrs['region'] = REGIONS.get(region, region)
    da_ds[key].attrs['region_key'] = region
    da_ds[key].attrs['variable'] = var
    da_ds[key].attrs['frequency'] = freq
    da_ds[key].attrs['source'] = build_url(var, region, freq)

    combined = da_ds if combined is None else combined.merge(da_ds, join='outer')

if combined is not None:
    combined.attrs['title'] = 'OSI SAF Sea Ice Index v3.0 (OSI-420-a) -- combined Arctic regions'
    combined.attrs['product'] = 'OSI-420-a, doi:10.15770/EUM_SAF_OSI_0022'
    combined.attrs['note'] = ('Internal variable naming per file was auto-detected, not '
                              'hardcoded -- see log for what was actually found per file.')

    nc_path = PROCESSED_DIR / 'osisaf_sii_v3_all_regions.nc'
    combined.to_netcdf(nc_path)
    logger.info(f"Combined NetCDF saved: {nc_path} ({len(combined.data_vars)} variables)")

combined


INFO:__main__:[ice_extent_glb_daily] 15 data variables present (['time_bounds', 'lat_bounds', 'lon_bounds', 'area', 'sie', 'source', 'rank_per_doy', 'min_per_doy', 'max_per_doy', 'yearly_min_rank', 'yearly_max_rank', 'yearly_min_date', 'yearly_max_date', 'yearly_min_value', 'yearly_max_value']); selected 'sie' as the primary quantity.
INFO:__main__:[ice_extent_glb_monthly] 9 data variables present (['time_bounds', 'lat_bounds', 'lon_bounds', 'area', 'sie', 'rank_per_month', 'absolute_trend', 'relative_trend_1981_2010', 'relative_trend_1991_2020']); selected 'sie' as the primary quantity.
INFO:__main__:[ice_extent_nh_daily] 15 data variables present (['time_bounds', 'lat_bounds', 'lon_bounds', 'area', 'sie', 'source', 'rank_per_doy', 'min_per_doy', 'max_per_doy', 'yearly_min_rank', 'yearly_max_rank', 'yearly_min_date', 'yearly_max_date', 'yearly_min_value', 'yearly_max_value']); selected 'sie' as the primary quantity.
INFO:__main__:[ice_extent_nh_monthly] 9 data variables present (['tim

<xarray.Dataset> Size: 6MB
Dimensions:                        (time: 17476)
Coordinates:
  * time                           (time) datetime64[ns] 140kB 1978-01-16 ......
Data variables: (12/40)
    ice_extent_glb_daily           (time) float64 140kB nan nan ... 22.99 nan
    ice_extent_glb_monthly         (time) float64 140kB nan nan nan ... nan nan
    ice_extent_nh_daily            (time) float64 140kB nan nan ... 6.088 nan
    ice_extent_nh_monthly          (time) float64 140kB nan nan nan ... nan nan
    ice_extent_baffin_daily        (time) float64 140kB nan nan ... 0.08666 nan
    ice_extent_baffin_monthly      (time) float64 140kB nan nan nan ... nan nan
    ...                             ...
    ice_extent_japan_daily         (time) float64 140kB nan nan nan ... 0.0 nan
    ice_extent_japan_monthly       (time) float64 140kB nan nan nan ... nan nan
    ice_extent_okhotsk_daily       (time) float64 140kB nan nan nan ... 0.0 nan
    ice_extent_okhotsk_monthly     (time) float64 140kB nan nan nan ... nan nan
    ice_extent_sval_daily          (time) float64 140kB nan nan ... 0.2651 nan
    ice_extent_sval_monthly        (time) float64 140kB nan nan nan ... nan nan
Attributes:
    title:    OSI SAF Sea Ice Index v3.0 (OSI-420-a) -- combined Arctic regions
    product:  OSI-420-a, doi:10.15770/EUM_SAF_OSI_0022
    note:     Internal variable naming per file was auto-detected, not hardco...

## Section 5 - Methodology writeup

In [ ]:
def write_methodology():
    lines = [
        "# OSI SAF Sea Ice Index v3.0 (OSI-420-a) -- Arctic Regions Download",
        "",
        "## Migration note from v2.3",
        "Greenland Sea region key changed: 'fram' (v2.3, used in figure_01) -> ",
        "'greenland' (v3.0, 'East Greenland Sea'). Update figure_01's fetch_sie() ",
        "call if switching that notebook to v3 data.",
        "",
        "## Regions downloaded",
    ]
    for key, name in REGIONS.items():
        lines.append(f"- **{key}**: {name}")
    lines += [
        "",
        "## Variables / frequencies",
        f"Variables: {VARIABLES}",
        f"Frequencies: {FREQUENCIES}",
        "",
        "## Status",
        f"{sum(1 for v in downloaded.values() if v is not None)} / {len(downloaded)} "
        f"files downloaded successfully.",
        "",
        "## Known caveats",
        "- ice_area file paths are inferred by naming-pattern consistency with the",
        "  confirmed ice_extent paths, not confirmed via an explicit data-table link.",
        "- Internal NetCDF variable names were not verified ahead of time (thredds.met.no",
        "  unreachable from the drafting environment) -- the loader auto-detects and logs",
        "  whatever variable(s) it finds per file; check osisaf_sii_v3.log.",
        "- Source: https://osisaf-hl.met.no/v3p0-sea-ice-index",
        "- Product citation: EUMETSAT OSI SAF, Sea-Ice Index (2025), OSI-420, "
        "doi:10.15770/EUM_SAF_OSI_0022",
    ]
    with open(METHODS_DIR / 'osisaf_sii_v3_methodology.md', 'w') as f:
        f.write('\n'.join(lines))
    logger.info(f"Methodology written: {METHODS_DIR / 'osisaf_sii_v3_methodology.md'}")

write_methodology()
